# Stigma Classifier — PEFT (LoRA) Fine-Tuning of Llama / Qwen — Two-Stage

**Stage 1:** train on the original dataset's `train.csv` / `test.csv` splits
(train further split into train/dev), evaluated on `test.csv` and on your own
test set (`golden_all_batches.xlsx`).

**Stage 2:** continue fine-tuning the stage-1 adapter on `golden_all_batches.xlsx`
itself, in 5-fold CV — each fold reloads the stage-1 adapter fresh (not the
previous fold's adapter), fine-tunes on 4/5 of your data, and evaluates on the
held-out 1/5.

**Runtime → Change runtime type → A100 GPU recommended.**

If using a Llama checkpoint, you need a Hugging Face account with access to the
gated `meta-llama` repo, and a token with read access.

In [ ]:
!pip install -q -U transformers datasets scikit-learn peft bitsandbytes accelerate

In [ ]:
from google.colab import files
uploaded = files.upload()  # select train.csv, test.csv, golden_all_batches.xlsx together

In [ ]:
from huggingface_hub import login
login()  # only needed for gated repos, e.g. meta-llama

In [ ]:
import gc
import ast
import torch
import pandas as pd
import random
import numpy as np
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, cohen_kappa_score
from sklearn.model_selection import train_test_split
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('social_stigma')

print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# ── Base model choice ───────────────────────────────────────────────────────
MODEL_CHOICE = 'qwen'  # 'qwen' or 'llama'
INSTRUCT     = True

MODEL_IDS = {
    'qwen':  {'instruct': 'Qwen/Qwen2.5-7B-Instruct',        'base': 'Qwen/Qwen2.5-7B'},
    'llama': {'instruct': 'meta-llama/Llama-3.1-8B-Instruct', 'base': 'meta-llama/Llama-3.1-8B'},
}
model_id = MODEL_IDS[MODEL_CHOICE]['instruct' if INSTRUCT else 'base']
print('Using base model:', model_id, '| instruct formatting:', INSTRUCT)

In [ ]:
# ── Labels ────────────────────────────────────────────────────────────────────
label_dict      = {'no': 0, 'yes': 1}
id2label_custom = {v: k for k, v in label_dict.items()}

STIGMATIZED_LABELS = {
    'Stigmatized (responsibility)',
    'Stigmatized (social distance)',
    'Stigmatized (fear/dangerousness)',
    'Stigmatized (anger)',
    'Stigmatized (coercion segregation)',
    'Stigmatized (helping)',
    'Stigmatized (pity)',
}

def map_to_binary(original_label: str) -> int:
    if original_label in STIGMATIZED_LABELS or original_label == 'Stigmatized':
        return label_dict['yes']
    return label_dict['no']

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
train_df  = pd.read_csv('train.csv')
test_df   = pd.read_csv('test.csv')
mytest_df = pd.read_excel('golden_all_batches.xlsx')[['text', 'stigma_present']].rename(columns={'stigma_present': 'label'})
mytest_df['label'] = mytest_df['label'].str.strip().str.lower()

print('Train size:  ', len(train_df))
print('Test size:   ', len(test_df))
print('My test size:', len(mytest_df))

In [ ]:
# ── Train / dev split (stratified on the binarized label) ──────────────────────
train_df['binary_label'] = train_df['label'].apply(map_to_binary)

train_idx, dev_idx = train_test_split(
    train_df.index, test_size=0.1, random_state=42, stratify=train_df['binary_label']
)
train_split_df = train_df.loc[train_idx].reset_index(drop=True)
dev_split_df   = train_df.loc[dev_idx].reset_index(drop=True)

print('Train split:', len(train_split_df))
print('Dev split:  ', len(dev_split_df))

In [ ]:
# ── Tokenizer + instruction formatting ─────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

SYSTEM_PROMPT = (
    "You are a classifier that determines whether a piece of text expresses or "
    "conveys stigma toward people with mental health conditions. Read the text "
    "and judge whether stigmatizing framing, language, or attitudes are present."
)

def conversation_to_text(conv) -> str:
    conv = ast.literal_eval(conv)
    return ' '.join(f"{msg['role']}: {msg['content']}" for msg in conv)

def format_prompt(text):
    if not INSTRUCT:
        return text
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Text: """{text}"""'},
    ]
    # add_generation_prompt=True appends the assistant turn header, so the
    # classification head pools a representation conditioned on "about to answer"
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    pos = label_dict['yes']
    return {
        'f1_binary':        f1_score(labels, predictions, pos_label=pos, average='binary'),
        'precision_binary': precision_score(labels, predictions, pos_label=pos, average='binary'),
        'recall_binary':    recall_score(labels, predictions, pos_label=pos, average='binary'),
        'f1_macro':         f1_score(labels, predictions, average='macro'),
        'precision_macro':  precision_score(labels, predictions, average='macro'),
        'recall_macro':     recall_score(labels, predictions, average='macro'),
        'accuracy':         accuracy_score(labels, predictions),
        'cohen_kappa':      cohen_kappa_score(labels, predictions),
    }

In [ ]:
# ── Determine MAX_LENGTH from the actual data ───────────────────────────────────
_all_texts = (
    [conversation_to_text(c) for c in train_df['conversations']]
    + [conversation_to_text(c) for c in test_df['conversations']]
    + mytest_df['text'].tolist()
)
lengths = [len(tokenizer(format_prompt(t), truncation=False)['input_ids']) for t in _all_texts]
print('max:', max(lengths), '| 99th pct:', int(np.percentile(lengths, 99)))

MAX_LENGTH = int(np.ceil(np.percentile(lengths, 99) / 8) * 8)  # adjust manually if needed

# Check whether truncation skews by label
_all_bin_labels = (
    train_df['label'].apply(map_to_binary).tolist()
    + test_df['label'].apply(map_to_binary).tolist()
    + mytest_df['label'].map(label_dict).tolist()
)
truncated = np.array(lengths) > MAX_LENGTH
print('truncated:', truncated.sum(), '/', len(lengths))
print(pd.Series(np.array(_all_bin_labels)[truncated]).value_counts(normalize=True))
print(pd.Series(_all_bin_labels).value_counts(normalize=True))

In [ ]:
# ── QLoRA setup ───────────────────────────────────────────────────────────────
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = compute_dtype,
    bnb_4bit_use_double_quant = True,
)

lora_config = LoraConfig(
    task_type        = TaskType.SEQ_CLS,
    r                = 16,
    lora_alpha       = 32,
    lora_dropout     = 0.05,
    bias             = 'none',
    target_modules   = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                         'gate_proj', 'up_proj', 'down_proj'],
)

def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels          = 2,
        quantization_config = bnb_config,
        device_map           = 'auto',
        torch_dtype          = compute_dtype,
    )
    model.config.id2label = id2label_custom
    model.config.label2id = label_dict
    model.config.pad_token_id = tokenizer.pad_token_id

    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

model = build_model()

In [ ]:
# ── Build tokenized datasets ─────────────────────────────────────────────────────
def encode_conv_examples(examples):
    texts = [format_prompt(conversation_to_text(c)) for c in examples['conversations']]
    return tokenizer(texts, max_length=MAX_LENGTH, padding='max_length', truncation=True)

def encode_plain_examples(examples):
    texts = [format_prompt(t) for t in examples['text']]
    return tokenizer(texts, max_length=MAX_LENGTH, padding='max_length', truncation=True)

def encode_label_finegrained(example):
    example['label'] = map_to_binary(example['label'])
    return example

def encode_label_yesno(example):
    example['label'] = label_dict[example['label']]
    return example

train_ds  = Dataset.from_pandas(train_split_df[['conversations', 'label']]).map(encode_conv_examples, batched=True).map(encode_label_finegrained)
dev_ds    = Dataset.from_pandas(dev_split_df[['conversations', 'label']]).map(encode_conv_examples, batched=True).map(encode_label_finegrained)
test_ds   = Dataset.from_pandas(test_df[['conversations', 'label']]).map(encode_conv_examples, batched=True).map(encode_label_finegrained)
mytest_ds = Dataset.from_pandas(mytest_df[['text', 'label']]).map(encode_plain_examples, batched=True).map(encode_label_yesno)

In [ ]:
# ── Stage 1 training (train.csv / dev split) ────────────────────────────────────
train_arg = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 16,
    gradient_checkpointing      = True,
    seed                        = 42,
    learning_rate               = 2e-4,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    optim                       = 'paged_adamw_8bit',
    eval_steps                  = 50,
    save_strategy               = 'steps',
    save_steps                  = 50,
    eval_strategy               = 'steps',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_binary',
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = train_arg,
    train_dataset   = train_ds,
    eval_dataset    = dev_ds,
    compute_metrics = compute_metrics,
)

trainer.train()

In [ ]:
# ── Stage 1 evaluation: original test.csv AND your own test set (zero-shot) ─────
def evaluate_and_report(dataset, name):
    preds    = trainer.predict(dataset)
    pred_ids = preds.predictions.argmax(axis=1)
    true_ids = dataset['label']
    pos      = label_dict['yes']

    print(f'=== {name} ===')
    print(f"Accuracy:            {accuracy_score(true_ids, pred_ids):.4f}")
    print(f"Precision (binary):  {precision_score(true_ids, pred_ids, pos_label=pos, average='binary'):.4f}")
    print(f"Recall    (binary):  {recall_score(true_ids, pred_ids, pos_label=pos, average='binary'):.4f}")
    print(f"F1        (binary):  {f1_score(true_ids, pred_ids, pos_label=pos, average='binary'):.4f}")
    print(f"Precision (macro):   {precision_score(true_ids, pred_ids, average='macro'):.4f}")
    print(f"Recall    (macro):   {recall_score(true_ids, pred_ids, average='macro'):.4f}")
    print(f"F1        (macro):   {f1_score(true_ids, pred_ids, average='macro'):.4f}")
    print(f"Cohen Kappa:         {cohen_kappa_score(true_ids, pred_ids):.4f}")
    print()
    return pred_ids

test_pred_ids   = evaluate_and_report(test_ds, 'Original test.csv')
mytest_pred_ids = evaluate_and_report(mytest_ds, 'My test set (golden_all_batches.xlsx)')

In [ ]:
# ── Save stage 1 predictions and download ───────────────────────────────────────
test_out_df = test_df.copy()
test_out_df['pred_label'] = [id2label_custom[p] for p in test_pred_ids]
test_out_df.to_csv('test_predictions.csv', index=False, encoding='utf-8')

mytest_out_df = mytest_df.copy()
mytest_out_df['pred_label'] = [id2label_custom[p] for p in mytest_pred_ids]
mytest_out_df.to_csv('mytest_predictions.csv', index=False, encoding='utf-8')

files.download('test_predictions.csv')
files.download('mytest_predictions.csv')  # this is the zero-shot (pre-stage-2) prediction on your data

In [ ]:
# ── Save the stage 1 adapter ─────────────────────────────────────────────────────
STAGE1_ADAPTER_DIR = './stage1_lora_adapter'
model.save_pretrained(STAGE1_ADAPTER_DIR)
print('Stage 1 adapter saved to', STAGE1_ADAPTER_DIR)

del model, trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ── Stage 2: continued fine-tuning on golden_all_batches.xlsx, 5-fold CV ─────────
from peft import PeftModel
from sklearn.model_selection import StratifiedKFold

def build_stage2_model():
    """Fresh quantized base + the STAGE 1 adapter reloaded from disk (not the
    previous fold's adapter — every fold starts from the same stage-1 point)."""
    base = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels          = 2,
        quantization_config = bnb_config,
        device_map           = 'auto',
        torch_dtype          = compute_dtype,
    )
    base.config.id2label = id2label_custom
    base.config.label2id = label_dict
    base.config.pad_token_id = tokenizer.pad_token_id
    base = prepare_model_for_kbit_training(base)

    model = PeftModel.from_pretrained(base, STAGE1_ADAPTER_DIR, is_trainable=True)
    model.print_trainable_parameters()
    return model

In [ ]:
mytest_df = mytest_df.reset_index(drop=True)
mytest_df['label_id'] = mytest_df['label'].map(label_dict)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = mytest_df['text'].values
y = mytest_df['label_id'].values

stage2_fold_results = []
stage2_all_preds    = np.zeros(len(mytest_df), dtype=int)
stage2_all_true     = np.zeros(len(mytest_df), dtype=int)

for fold, (train_val_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f'\n=== Stage 2 — Fold {fold+1}/5 ===')

    # 4/5 train_val further split into train (80%) + val (20%) for eval_strategy
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    inner_skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_sub_idx, val_sub_idx = next(inner_skf.split(X_train_val, y_train_val))
    train_idx_final = train_val_idx[train_sub_idx]
    val_idx_final   = train_val_idx[val_sub_idx]

    fold_train_df = mytest_df.iloc[train_idx_final].reset_index(drop=True)
    fold_val_df   = mytest_df.iloc[val_idx_final].reset_index(drop=True)
    fold_test_df  = mytest_df.iloc[test_idx].reset_index(drop=True)

    def to_ds(subset):
        ds = Dataset.from_pandas(subset[['text', 'label_id']].rename(columns={'label_id': 'label'}))
        return ds.map(encode_plain_examples, batched=True)

    fold_train_ds = to_ds(fold_train_df)
    fold_val_ds   = to_ds(fold_val_df)
    fold_test_ds  = to_ds(fold_test_df)

    model = build_stage2_model()

    stage2_train_arg = TrainingArguments(
        output_dir                  = f'./results/stage2_fold_{fold+1}',
        num_train_epochs            = 5,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size  = 4,
        gradient_accumulation_steps = 16,
        gradient_checkpointing      = True,
        seed                        = 42,
        learning_rate               = 1e-4,
        bf16                        = torch.cuda.is_bf16_supported(),
        fp16                        = not torch.cuda.is_bf16_supported(),
        optim                       = 'paged_adamw_8bit',
        eval_steps                  = 20,
        save_strategy               = 'steps',
        save_steps                  = 20,
        eval_strategy               = 'steps',
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1_binary',
        report_to                   = 'none',
    )

    stage2_trainer = Trainer(
        model           = model,
        args            = stage2_train_arg,
        train_dataset   = fold_train_ds,
        eval_dataset    = fold_val_ds,
        compute_metrics = compute_metrics,
    )

    stage2_trainer.train()

    preds    = stage2_trainer.predict(fold_test_ds)
    pred_ids = preds.predictions.argmax(axis=1)
    true_ids = fold_test_ds['label']

    stage2_all_preds[test_idx] = pred_ids
    stage2_all_true[test_idx]  = true_ids

    pos = label_dict['yes']
    fold_metrics = {
        'fold':             fold + 1,
        'f1_binary':        f1_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'precision_binary': precision_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'recall_binary':    recall_score(true_ids, pred_ids, pos_label=pos, average='binary'),
        'f1_macro':         f1_score(true_ids, pred_ids, average='macro'),
        'precision_macro':  precision_score(true_ids, pred_ids, average='macro'),
        'recall_macro':     recall_score(true_ids, pred_ids, average='macro'),
        'accuracy':         accuracy_score(true_ids, pred_ids),
        'cohen_kappa':      cohen_kappa_score(true_ids, pred_ids),
    }
    stage2_fold_results.append(fold_metrics)
    print(fold_metrics)

    del model, stage2_trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ── Stage 2 aggregate results ─────────────────────────────────────────────────
stage2_results_df = pd.DataFrame(stage2_fold_results)
print('=== Stage 2 — per-fold results ===')
print(stage2_results_df.to_string(index=False))

metric_cols = [c for c in stage2_results_df.columns if c != 'fold']
print('\n=== Stage 2 — mean ± std across folds ===')
for col in metric_cols:
    print(f"{col:20s}: {stage2_results_df[col].mean():.4f} ± {stage2_results_df[col].std():.4f}")

pos = label_dict['yes']
print('\n=== Stage 2 — overall (all folds combined, out-of-fold) ===')
print(f"Accuracy:            {accuracy_score(stage2_all_true, stage2_all_preds):.4f}")
print(f"Precision (binary):  {precision_score(stage2_all_true, stage2_all_preds, pos_label=pos, average='binary'):.4f}")
print(f"Recall    (binary):  {recall_score(stage2_all_true, stage2_all_preds, pos_label=pos, average='binary'):.4f}")
print(f"F1        (binary):  {f1_score(stage2_all_true, stage2_all_preds, pos_label=pos, average='binary'):.4f}")
print(f"Precision (macro):   {precision_score(stage2_all_true, stage2_all_preds, average='macro'):.4f}")
print(f"Recall    (macro):   {recall_score(stage2_all_true, stage2_all_preds, average='macro'):.4f}")
print(f"F1        (macro):   {f1_score(stage2_all_true, stage2_all_preds, average='macro'):.4f}")
print(f"Cohen Kappa:         {cohen_kappa_score(stage2_all_true, stage2_all_preds):.4f}")
print('\nCompare this to the stage 1 (zero-shot) numbers on golden_all_batches.xlsx above.')

In [ ]:
# ── Save stage 2 predictions and download ────────────────────────────────────
stage2_out_df = mytest_df[['text', 'label']].copy()
stage2_out_df['pred_label'] = [id2label_custom[p] for p in stage2_all_preds]
stage2_out_df.to_csv('stage2_kfold_predictions.csv', index=False, encoding='utf-8')

stage2_results_df.to_csv('stage2_kfold_metrics.csv', index=False)

files.download('stage2_kfold_predictions.csv')
files.download('stage2_kfold_metrics.csv')